Installing libraries:

transformers, accelerate, bitsandbytes, torch: for loading and running deep seek models efficintly.

datasets: To load financial news data.

gradio: for building UI

In [1]:
print("Installing libraries...")
!pip install -q transformers accelerate bitsandbytes>=0.46.1 datasets torch gradio
print("successfully installed!")
print("Please restart the runtime after this installation to ensure bitsandbytes is properly loaded.")

Installing libraries...
successfully installed!
Please restart the runtime after this installation to ensure bitsandbytes is properly loaded.


In [2]:
#hugging face login
import os
from huggingface_hub import login, notebook_login
print("logging in hugging face ...")
notebook_login()
print("successfully logged in!")

logging in hugging face ...
successfully logged in!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from datasets import load_dataset #import the dataset loading function
import gradio as gr
from IPython.display import display, Markdown
import random #to pick random news item

#check GPU availability
if torch.cuda.is_available():
  print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
  torch.set_default_device("cuda")
  print("pytorch default device set to cuda(GPU).")
else:
  print("No GPU detected. Running on CPU")

GPU Detected: Tesla T4
pytorch default device set to cuda(GPU).


In [4]:
#helper function for markdown display
def print_markdown(text):
  """Displays text as markdown"""
  display(Markdown(text))

**TASK: Loading financial news datasets from hugging face**


1.   Use load_dataset("dataset_name") to download and cache the dataset
2.   inspect the dataset structure and features


1.  select relavant columns(title and description) for analysis
2.  combine the title and description into a single text input to the llm





In [5]:
dataset_id = "mohit9999/all_news_finance_sm_1h2023_custom"
print(f"Loading dataset:{dataset_id}")
#load the dataset
#we need only 'train' split, specify split = 'train' if needed
#The datatype of news_dataset is datasets.Datasets
news_dataset = load_dataset(dataset_id, split = 'train')
print("dataset loaded successfully")

Loading dataset:mohit9999/all_news_finance_sm_1h2023_custom
dataset loaded successfully


In [6]:
#inspecting the dataset
news_dataset

Dataset({
    features: ['instruction', 'output', 'text'],
    num_rows: 5062
})

In [7]:
#displaying features
print("\n Dataset Features:")
print(news_dataset.features)


 Dataset Features:
{'instruction': Value('string'), 'output': Value('string'), 'text': Value('string')}


In [8]:
#view datasets in pandas dataframe
news_dataset.to_pandas()

,instruction,output,text
0,"Dow drops 400 points, turns negative for the y...",Regional banks led the broader market lower as...,Below is an instruction that describes a task....
1,Bitcoin drops with stocks as analyst warns of ...,Bitcoin dips as the U.S. banking crisis engulf...,Below is an instruction that describes a task....
2,Bitcoin Price Analysis: 29370 Tested After Su...,Bitcoin (BTC/USD) sought to add to recent gain...,Below is an instruction that describes a task....
3,"Bitcoin Is 75% To Halving, Here's How Past Cyc...",The current Bitcoin cycle is now 75% on the wa...,Below is an instruction that describes a task....
4,"Iron Mountain FFO of $0.71 beats by $0.03, rev...",Iron Mountain press release (IRM): Q1 FFO of...,Below is an instruction that describes a task....
...,...,...,...
5057,"Ethereum Merge can trigger high volatility, Bi...",BitMEX CEO Alexander Höptner said that their t...,Below is an instruction that describes a task....
5058,EXPLAINER: Is COVID-19 winding down? Scientist...,Is the coronavirus on its way out? You might t...,Below is an instruction that describes a task....
5059,How Bitcoin Educates The World About Finance,Bitcoin brings financial education to the fore...,Below is an instruction that describes a task....
5060,Crypto Hedge Fund Three Arrows Capital Wallet ...,The removal of liquidity was the first activit...,Below is an instruction that describes a task....


In [9]:
#preparing data for llm
#combine title and description for input text
def combine_news_text(example):
  title = example.get("instruction","") or ""
  description = example.get("output","") or ""
  return {"full_text": f"Title:{title}\nDescription{description}"}

In [10]:
#applying the function
#this uses maps which is efficient for datasets
news_dataset = news_dataset.map(combine_news_text)
print("\n- - - sample data with full text- - -")
print(news_dataset[0])



- - - sample data with full text- - -
{'instruction': 'Dow drops 400 points, turns negative for the year as bank fears grow: Live updates', 'output': 'Regional banks led the broader market lower as contagion fears resurfaced.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction: Dow drops 400 points, turns negative for the year as bank fears grow: Live updates\n\n### Response: Regional banks led the broader market lower as contagion fears resurfaced.', 'full_text': 'Title:Dow drops 400 points, turns negative for the year as bank fears grow: Live updates\nDescriptionRegional banks led the broader market lower as contagion fears resurfaced.'}


In [11]:
#display the full text of first sample
print(news_dataset[0]["full_text"])

Title:Dow drops 400 points, turns negative for the year as bank fears grow: Live updates
DescriptionRegional banks led the broader market lower as contagion fears resurfaced.


In [12]:
display(news_dataset.select(range(10)).to_pandas())

,instruction,output,text,full_text
0,"Dow drops 400 points, turns negative for the y...",Regional banks led the broader market lower as...,Below is an instruction that describes a task....,"Title:Dow drops 400 points, turns negative for..."
1,Bitcoin drops with stocks as analyst warns of ...,Bitcoin dips as the U.S. banking crisis engulf...,Below is an instruction that describes a task....,Title:Bitcoin drops with stocks as analyst war...
2,Bitcoin Price Analysis: 29370 Tested After Su...,Bitcoin (BTC/USD) sought to add to recent gain...,Below is an instruction that describes a task....,Title:Bitcoin Price Analysis: 29370 Tested Af...
3,"Bitcoin Is 75% To Halving, Here's How Past Cyc...",The current Bitcoin cycle is now 75% on the wa...,Below is an instruction that describes a task....,"Title:Bitcoin Is 75% To Halving, Here's How Pa..."
4,"Iron Mountain FFO of $0.71 beats by $0.03, rev...",Iron Mountain press release (IRM): Q1 FFO of...,Below is an instruction that describes a task....,Title:Iron Mountain FFO of $0.71 beats by $0.0...
5,"Moderna GAAP EPS of $0.19 beats by $1.94, reve...",Moderna press release (MRNA): Q1 GAAP EPS of...,Below is an instruction that describes a task....,Title:Moderna GAAP EPS of $0.19 beats by $1.94...
6,Regeneron Pharmaceuticals Non-GAAP EPS of $10....,Regeneron Pharmaceuticals press release (REGN...,Below is an instruction that describes a task....,Title:Regeneron Pharmaceuticals Non-GAAP EPS o...
7,Apple earnings likely to show solid iPhone res...,Apple (AAPL) is set to report fiscal second-qu...,Below is an instruction that describes a task....,Title:Apple earnings likely to show solid iPho...
8,Rug Pulls and Meme Coins: Lessons from XIRTAM’...,The recent rug pull by XIRTAM raises concerns ...,Below is an instruction that describes a task....,Title:Rug Pulls and Meme Coins: Lessons from X...
9,Bitcoin Price Analysis: Stops Reached Below 28...,BTC/USD Stops Elected Below 28164: Sally Ho’s...,Below is an instruction that describes a task....,Title:Bitcoin Price Analysis: Stops Reached Be...


In [13]:
#accessing the 100th news in dataset
print(news_dataset[99]['full_text'])

Title:Stock futures fall as investors digest latest earnings, higher rates: Live updates
DescriptionInvestors kept their eyes on key earnings releases. Morgan Stanley and Tesla are due to report Wednesday.


**load and test Deepseek reasoning models**

In [14]:
# Install a compatible PyTorch version with CUDA support to resolve version conflicts
# and ensure proper GPU utilization.
print("Installing compatible PyTorch...")
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print("PyTorch installed successfully!")


Installing compatible PyTorch...
Looking in indexes: https://download.pytorch.org/whl/cu121
PyTorch installed successfully!


In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#choosing deepseek model
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B" # Changed model to 14B
print(f"loading model:{model_id}")
print(f"this may take a few minutes...")

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=bnb_config, # Pass the quantization config
                                             device_map="auto",
                                             trust_remote_code=True,
                                             torch_dtype=torch.bfloat16 # Ensure model weights are loaded with bfloat16
)
model.eval()
print("Model loaded successfully in 4-bit")

loading model:deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
this may take a few minutes...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully in 4-bit


In [16]:
#loading Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id,
                                        trust_remote_code = True)

In [17]:
#creating text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    torch_dtype = "auto",
    device_map = "auto")
print("text-generation pipeline created!")

`torch_dtype` is deprecated! Use `dtype` instead!


text-generation pipeline created!


In [18]:
#building full formatted prompt
#test questions
test_question = "what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short"
#test_question = "Explain what are electric cars in simple terms.keep thinking short."
test_prompt = f"<|im_start|>user\n{test_question}\n<|im_end|>\n<|im_start|>assistant\n<think>\n"
print(f"Test prompt:\n{test_prompt}")

Test prompt:
<|im_start|>user
what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short
<|im_end|>
<|im_start|>assistant
<think>



In [20]:
#testing the model
#generate the text using pipeline
print("testing model")

outputs = llm_pipeline(test_prompt,
                       max_new_tokens =4000,
                       do_sample = True,
                       temperature = 0.7,
                       top_p = 0.9)
outputs

Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


testing model


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "<|im_start|>user\nwhat is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short\n<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I have this problem where I initially have 5 apples. Then I give 2 to my mom and another 2 to my sister. I need to figure out how many apples I have now. Let me break it down step by step.\n\nFirst, I start with 5 apples. That's straightforward. Now, I give away 2 apples to my mom. So, I subtract 2 from my current count. That would be 5 minus 2, which equals 3 apples. Now, I have 3 apples left.\n\nNext, I give another 2 apples to my sister. So, I subtract another 2 from my current count. That would be 3 minus 2, which equals 1 apple. Now, I have only 1 apple left.\n\nWait, but the problem says to keep the thinking short. Maybe there's a simpler way to look at it. I initially had 5, then I gave away a total of 4 apples (2 to my mom and 2 to my sister). S

In [21]:
#function to extract and format the reason and output
def format_model_output(output):
  reason_start = output.find("<think>") + len("<think>")
  reason_end = output.find("</think>")
  reason = output[reason_start:reason_end].strip()

  output_start = reason_end + len("</think>")
  model_output_content = output[output_start:].strip()

  reason = f"**Reason**:\n{reason}\n"
  output = f"**Output**:\n{output}\n"
  return reason, output

In [22]:
#extract and print generated text
full_output = outputs[0]['generated_text']
reason, output = format_model_output(full_output)
print_markdown(reason)
print_markdown(output)

**Reason**:
Okay, so I have this problem where I initially have 5 apples. Then I give 2 to my mom and another 2 to my sister. I need to figure out how many apples I have now. Let me break it down step by step.

First, I start with 5 apples. That's straightforward. Now, I give away 2 apples to my mom. So, I subtract 2 from my current count. That would be 5 minus 2, which equals 3 apples. Now, I have 3 apples left.

Next, I give another 2 apples to my sister. So, I subtract another 2 from my current count. That would be 3 minus 2, which equals 1 apple. Now, I have only 1 apple left.

Wait, but the problem says to keep the thinking short. Maybe there's a simpler way to look at it. I initially had 5, then I gave away a total of 4 apples (2 to my mom and 2 to my sister). So, I subtract 4 from 5, which gives me 1 apple. That seems to align with what I calculated earlier.

I think that's all there is to it. The answer is 1 apple.


**Output**:
<|im_start|>user
what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short
<|im_end|>
<|im_start|>assistant
<think>
Okay, so I have this problem where I initially have 5 apples. Then I give 2 to my mom and another 2 to my sister. I need to figure out how many apples I have now. Let me break it down step by step.

First, I start with 5 apples. That's straightforward. Now, I give away 2 apples to my mom. So, I subtract 2 from my current count. That would be 5 minus 2, which equals 3 apples. Now, I have 3 apples left.

Next, I give another 2 apples to my sister. So, I subtract another 2 from my current count. That would be 3 minus 2, which equals 1 apple. Now, I have only 1 apple left.

Wait, but the problem says to keep the thinking short. Maybe there's a simpler way to look at it. I initially had 5, then I gave away a total of 4 apples (2 to my mom and 2 to my sister). So, I subtract 4 from 5, which gives me 1 apple. That seems to align with what I calculated earlier.

I think that's all there is to it. The answer is 1 apple.
</think>

First, you start with 5 apples.

Then, you give away 2 apples to your mom:  
5 - 2 = 3 apples.

Then, you give away another 2 apples to your sister:  
3 - 2 = 1 apple.

So, you have **1 apple** left.

**Answer:** 1 apple.


**PROMPTING FOR REASONING AND CLAASIFICATION**

This is the core of the project

Craft a prompt that tells the deepseek model

1. Analyze the provided financial news.

2. Think step-by-step from a financial perspective. why this news in positive negative or neutral.

3. output the reasoning

4. output the final classification.

using clear labels in the prompt helps the model structure its output.

**steps**

1. define a detailed prompt.

2. create a function analyze_news_sentiment that:

takes news text and the llm pipeline.

formats the prompt.

calls the pipeline to generate response.

parses the generated text to seperate the reasoning from final classification.

3. test the function on a few examples.

In [20]:
# This cell previously defined 'analyze_news_sentiment'.
# Its content has been moved to cell 'vWG_WAOT9k1Y' to resolve scope issues with Gradio.

In [16]:
# This cell was a duplicate creation of llm_pipeline and is no longer needed. The llm_pipeline is created in cell YNCyStRBPh1c.

In [28]:
print("\n"+ "="* 30 + "TESTING ANALYSIS" + "=" * 30 )
#select a few random indices from dataset to test the analysis function
random_indices = random.sample(range(len(news_dataset)), 3)

#iterate over each randomly selected index
for index in random_indices:
  #retrive the full text of the news items in current index
  sample_news = news_dataset[index]["full_text"]

  #analyze the sentiment of the full text
  reason, output = analyze_news_sentiment(sample_news)

  #print the results
  print(f"\n - - - Analysis result for index{index} - - -")

  #display the reasoning and output
  print_markdown(reason)
  print_markdown(output)

#printing seperate line for better readabilty results
print("-" * 60)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



==============================TESTING ANALYSIS==============================
Raw output:
<|im_start|>user
Based on the following financial news, provide:
1. A brief financial reasoning (1-2 points)
2. Your sentiment classification (Positive, Negative, or Neutral)

News: Title:Fed Chair Powell says smaller interest rate hikes could start in December
DescriptionFederal Reserve Chairman Jerome Powell confirmed Wednesday that smaller interest rate increases are likely ahead.

Provide your response in this format:
Reasoning: [your analysis]
Sentiment: [Positive/Negative/Neutral]
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>user
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>user
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant

<|

**Sentiment:** Unknown

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw output:
<|im_start|>user
Based on the following financial news, provide:
1. A brief financial reasoning (1-2 points)
2. Your sentiment classification (Positive, Negative, or Neutral)

News: Title:Amazon will mehr als 18.000 Stellen streichen
DescriptionDie Entlassungswelle bei Amazon fällt größer aus als bislang bekannt war: Tausende Beschäftigte des kriselnden Konzerns verlieren ihren Job. Es ist der größte Stellenabbau der Unternehmensgeschichte.

Provide your response in this format:
Reasoning: [your analysis]
Sentiment: [Positive/Negative/Neutral]
<|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>sentiment analysis based on the news content>
<|im_end|>
<|im_start|>reasoning: the news mentions amazon reducing their number of positions due to a massive layoffs, leading to a significant loss of employees. the negative impact on the workforce is a strong indicator of a negative sentiment.
<|im_end|>
<|im_start|>sentiment: negative
<|im_end|>
<|im_start|>reasoning: the news de

No reasoning provided

**Sentiment:** <|im_start|>reasoning: the news details the layoffs at amazon, which resulted in a massive loss of employees. this negative impact on the workforce is a clear sign of

------------------------------------------------------------


building gradio interface


In [30]:
import gradio as gr
import random
import re
from transformers import pipeline

In [37]:
# import gradio as gr
# import random
# import re
# from transformers import pipeline

# def format_model_output(full_output):
#     """Parse the model output into reasoning and final sentiment, including <think> tags."""
#     # Find the last assistant turn to ensure we're parsing the latest response
#     assistant_start_marker = "<|im_start|>assistant\n<think>"
#     assistant_start_index = full_output.rfind(assistant_start_marker)

#     if assistant_start_index == -1:
#         # Fallback if the expected assistant start marker isn't found at all
#         assistant_turn_start = full_output.rfind("<|im_start|>assistant\n")
#         if assistant_turn_start != -1:
#             model_generated_content = full_output[assistant_turn_start + len("<|im_start|>assistant\n"):].strip()
#         else:
#             model_generated_content = full_output.strip()
#     else:
#         model_generated_content = full_output[assistant_start_index + len(assistant_start_marker):].strip()

#     # Find the end of the thought process, indicated by </think> or <|im_end|>
#     think_end_index = model_generated_content.find("</think>")
#     im_end_index = model_generated_content.find("<|im_end|>")

#     # Determine the actual end of the relevant generation
#     if think_end_index != -1 and (im_end_index == -1 or think_end_index < im_end_index):
#         parse_content = model_generated_content[:think_end_index].strip()
#     elif im_end_index != -1:
#         parse_content = model_generated_content[:im_end_index].strip()
#     else:
#         parse_content = model_generated_content.strip()

#     # Now parse the content for Reasoning and Sentiment using regex
#     reasoning_text_parts = []
#     extracted_sentiment = "Unknown"

#     # Using re.DOTALL to allow "." to match newlines if reasoning spans multiple lines
#     # Use lookahead to stop reasoning capture before Sentiment
#     reasoning_match = re.search(r"Reasoning:\s*(.*?)(?=\nSentiment:|$)", parse_content, re.DOTALL | re.IGNORECASE)
#     sentiment_match = re.search(r"Sentiment:\s*(.*?)(?=\n|$)", parse_content, re.DOTALL | re.IGNORECASE)

#     if reasoning_match:
#         reasoning_text_parts.append(reasoning_match.group(1).strip())

#     if sentiment_match:
#         extracted_sentiment = sentiment_match.group(1).strip()

#     final_reason = "\n".join(reasoning_text_parts).strip()
#     final_sentiment = f"**Sentiment:** {extracted_sentiment}"

#     return final_reason if final_reason else "No reasoning provided", final_sentiment


# def analyze_news_sentiment(news_text):
#     """Analyzes news text using DeepSeek model."""
#     question_template = """Based on the following financial news, provide:
# 1. A brief financial reasoning (1-2 points)
# 2. Your sentiment classification (Positive, Negative, or Neutral)

# News: {}

# Example Output Format:
# Reasoning: This news indicates that the Fed is likely to slow down interest rate hikes. This is generally seen as positive for the stock market as it reduces borrowing costs and can stimulate economic growth.
# Sentiment: Positive

# Please provide your response in this exact format:
# Reasoning: [Your financial reasoning]
# Sentiment: [Positive/Negative/Neutral]"""

#     test_question = question_template.format(news_text)

#     # Prime the assistant's response with <think> to guide structured generation
#     test_prompt = f"<|im_start|>user\n{test_question}\n<|im_end|>\n<|im_start|>assistant\n<think>\n"

#     try:
#         outputs = llm_pipeline(
#             test_prompt,
#             max_new_tokens=512,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.9
#         )

#         full_output = outputs[0]["generated_text"]

#         # Debug: print raw output
#         print(f"Raw output:\n{full_output}")

#         reason, output = format_model_output(full_output)

#         return reason, output

#     except Exception as e:
#         return f"Error: {str(e)}", "Error occurred"


# # Helper functions for Gradio (define if not already defined elsewhere and needed here)
# # Assuming news_dataset is globally available from previous cells

# def get_random_news():
#     """Fetches a random financial news item from the dataset."""
#     random_index = random.randint(0, len(news_dataset) - 1)
#     return news_dataset[random_index]["full_text"]

# def perform_analysis(news_text):
#     """Performs sentiment analysis on the provided news text."""
#     if not news_text:
#         return "Please fetch a news item first.", "**Sentiment:** N/A"
#     reason, sentiment_output = analyze_news_sentiment(news_text)
#     return reason, sentiment_output



# # Building Gradio interface - fixed
# with gr.Blocks() as demo:
#     gr.Markdown("""
#     # Deepseek Financial News Analyzer
#     Fetch random news and analyze sentiment with AI reasoning.
#     """)

#     with gr.Row():
#         btn_fetch = gr.Button("Fetch Random News", variant="secondary")
#         btn_analyze = gr.Button("Analyze News", variant="primary")

#     news_display = gr.Textbox(
#         label="News Text",
#         lines=6,
#         interactive=False
#     )

#     with gr.Tab("Reasoning"):
#         analysis_display = gr.Textbox(lines=10, show_label=False)

#     with gr.Tab("Result"):
#         analysis_output = gr.Markdown()

#     btn_fetch.click(
#         fn=get_random_news,
#         outputs=news_display
#     )

#     btn_analyze.click(
#         fn=perform_analysis,
#         inputs=news_display,
#         outputs=[analysis_display, analysis_output]
#     )

#     demo.load(
#         fn=get_random_news,
#         outputs=news_display
#     )

# demo.launch(share=True, server_name="0.0.0.0")

In [38]:
# The llm_pipeline is already initialized in cell YNCyStRBPh1c and should be reused.
# from transformers import pipeline
# import torch

# llm_pipeline = pipeline(
#     "text-generation",
#     model="deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
#     model_kwargs={"torch_dtype": torch.bfloat16},
#     device_map="auto"
# )


def format_model_output(full_output):
    """Parse model output to extract reasoning and sentiment."""
    import re

    # Extract content after the last assistant turn
    assistant_start_marker = "<|im_start|>assistant"
    parts = full_output.split(assistant_start_marker)
    model_response = parts[-1] if len(parts) > 1 else full_output

    # Remove the <|im_end|> tag if present
    model_response = model_response.split("<|im_end|>")[0].strip()

    # Initialize with default values
    reasoning = "No reasoning provided."
    sentiment = "Unknown"

    # Extract text within <think> tags for the reasoning part
    think_match = re.search(r"<think>(.*?)</think>", model_response, re.DOTALL)
    if think_match:
        think_content = think_match.group(1).strip()
        # Try to find Reasoning and Sentiment within the <think> block first
        reasoning_in_think = re.search(r"Reasoning:\s*(.*?)(?=\nSentiment:|$)", think_content, re.DOTALL | re.IGNORECASE)
        sentiment_in_think = re.search(r"Sentiment:\s*(.*?)(?=\n|$)", think_content, re.DOTALL | re.IGNORECASE)
        if reasoning_in_think:
            reasoning = reasoning_in_think.group(1).strip()
        if sentiment_in_think:
            sentiment = sentiment_in_think.group(1).strip()

        # Remove the <think> block from the main response to avoid double parsing
        model_response = re.sub(r"<think>.*?</think>", "", model_response, flags=re.DOTALL)

    # If not found in <think> block, or if <think> block didn't contain them, search in the rest of the response
    # This handles cases where the model puts the final answer outside <think>
    if reasoning == "No reasoning provided." or sentiment == "Unknown":
        reasoning_match = re.search(r"Reasoning:\s*(.*?)(?=\nSentiment:|$)", model_response, re.DOTALL | re.IGNORECASE)
        sentiment_match = re.search(r"Sentiment:\s*(.*?)(?=\n|$)", model_response, re.DOTALL | re.IGNORECASE)

        if reasoning_match and reasoning == "No reasoning provided.": # Only update if not already found in think block
            reasoning = reasoning_match.group(1).strip()
        if sentiment_match and sentiment == "Unknown": # Only update if not already found in think block
            sentiment = sentiment_match.group(1).strip()

    return reasoning, f"**Sentiment:** {sentiment}"


def analyze_news_sentiment(news_text):
    """Analyzes news text using the existing DeepSeek pipeline."""
    question_template = """Based on the following financial news, provide:
1. A brief financial reasoning (1-2 points)
2. Your sentiment classification (Positive, Negative, or Neutral)

News: {}

Example Output Format:
Reasoning: This news indicates that the Fed is likely to slow down interest rate hikes. This is generally seen as positive for the stock market as it reduces borrowing costs and can stimulate economic growth.
Sentiment: Positive

Please provide your response in this exact format:
Reasoning: [Your financial reasoning]
Sentiment: [Positive/Negative/Neutral]"""

    formatted_question = question_template.format(news_text)

    # Construct the prompt in the Deepseek chat format for text-generation pipeline
    test_prompt = f"<|im_start|>user\n{formatted_question}\n<|im_end|>\n<|im_start|>assistant\n<think>\n"

    try:
        outputs = llm_pipeline(
            test_prompt,
            max_new_tokens=512,
            do_sample=True, # Allow for more varied generation if needed
            temperature=0.7, # Increased temperature for more diverse generation
            top_p=0.9
        )

        full_output = outputs[0]["generated_text"]
        print(f"Raw output:\n{full_output}") # Debug

        return format_model_output(full_output)
    except Exception as e:
        return f"Error during analysis: {str(e)}", "**Sentiment:** Error"


# Helper functions for Gradio (make sure these are available or defined here)
def get_random_news():
    """Fetches a random financial news item from the dataset."""
    # news_dataset is assumed to be globally available from previous cells
    if 'news_dataset' not in globals() or len(news_dataset) == 0:
        return "Error: News dataset not loaded."
    random_index = random.randint(0, len(news_dataset) - 1)
    return news_dataset[random_index]["full_text"]

def perform_analysis(news_text):
    """Performs sentiment analysis on the provided news text."""
    if not news_text:
        return "Please fetch a news item first.", "**Sentiment:** N/A"
    reason, sentiment_output = analyze_news_sentiment(news_text)
    return reason, sentiment_output

# Building Gradio Interface
import gradio as gr
import random
import re # Ensure re is imported for format_model_output

with gr.Blocks() as demo:
    gr.Markdown("# Deepseek Financial News Analyzer")

    with gr.Row():
        btn_fetch = gr.Button("Fetch Random News", variant="secondary")
        btn_analyze = gr.Button("Analyze", variant="primary")

    news_display = gr.Textbox(label="News Text", lines=6, interactive=False)

    with gr.Tab("Reasoning"):
        reasoning_output = gr.Textbox(lines=10, show_label=False)

    with gr.Tab("Result"):
        sentiment_output = gr.Markdown()

    btn_fetch.click(
        fn=get_random_news,
        outputs=news_display
    )
    btn_analyze.click(
        fn=perform_analysis,
        inputs=news_display,
        outputs=[reasoning_output, sentiment_output]
    )
    demo.load(
        fn=get_random_news,
        outputs=news_display
    )

demo.launch(share=True, server_name="0.0.0.0")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://16f102fd50d82a5f19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
